In [14]:
import numpy as np
import pandas as pd

funda = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\1. Raw Data\funda.csv")
wx = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\1. Raw Data\weather_hist.csv")

In [15]:
funda["gasDayStart"] = pd.to_datetime(funda["gasDayStart"])
funda = funda.sort_values("gasDayStart")

feb29 = funda["gasDayStart"].dt.month.eq(2) & funda["gasDayStart"].dt.day.eq(29)
key = funda["gasDayStart"].dt.strftime("%m-%d").mask(feb29, "02-28")  # 29 Feb uses the 28 Feb normal
year = funda["gasDayStart"].dt.year

# 29 February stays out of the 28 February average
hist = funda.loc[~feb29, ["gasDayStart", "full"]].copy()
hist["key"] = hist["gasDayStart"].dt.strftime("%m-%d")
hist["year"] = hist["gasDayStart"].dt.year
lookup = hist.set_index(["key", "year"])["full"]

prior = pd.DataFrame({
    lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup)
    for lag in range(1, 6)
})
funda["storage_dev"] = funda["full"] - prior.mean(axis=1, skipna=False)  # five preceding years, or missing

In [16]:
funda["net_inj"] = -funda["netWithdrawal"] / 1000  # TWh/d, positive when injecting

stock_gap = funda["gasInStorage"].diff() - funda["net_inj"]
assert stock_gap.abs().median() < 0.1  # lines up with the stock change, inside revision noise

In [17]:
wx["date"] = pd.to_datetime(wx["date"])
wx = wx.dropna(subset=["EU_HDD", "EU_CDD"]).sort_values("date")
wx = wx.rename(columns={"EU_HDD": "hdd"})

feb29 = wx["date"].dt.month.eq(2) & wx["date"].dt.day.eq(29)
key = wx["date"].dt.strftime("%m-%d").mask(feb29, "02-28")  # 29 Feb uses the 28 Feb normal
year = wx["date"].dt.year

# 29 February stays out of the 28 February average
hist = wx.loc[~feb29, ["date", "hdd"]].copy()
hist["key"] = hist["date"].dt.strftime("%m-%d")
hist["year"] = hist["date"].dt.year
lookup = hist.set_index(["key", "year"])["hdd"]

prior = pd.DataFrame({
    lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup)
    for lag in range(1, 11)
})
wx["hdd_anom"] = wx["hdd"] - prior.mean(axis=1, skipna=False)  # daily anomaly, not the 7-day sum

In [18]:
funda["storage_dev_lag"] = funda["storage_dev"].shift(1)  # yesterday's stock, on the gas-day index
gas = funda[["gasDayStart", "net_inj", "storage_dev_lag"]].merge(
    wx[["date", "hdd_anom"]], left_on="gasDayStart", right_on="date", how="left"
)
gas = gas.drop(columns=["date"]).sort_values("gasDayStart").reset_index(drop=True)

In [19]:
# Earlier gas days only. January is the base. A column that does not vary in the past sample is left out.
month = gas["gasDayStart"].dt.month.to_numpy()
D = np.column_stack([
    np.ones(len(gas)),
    gas["hdd_anom"].to_numpy(),
    gas["storage_dev_lag"].to_numpy(),
    *[(month == m).astype(float) for m in range(2, 13)],
])
y = gas["net_inj"].to_numpy()
complete = np.flatnonzero(np.isfinite(D).all(axis=1) & np.isfinite(y))
resid = np.full(len(gas), np.nan)

for j in range(len(complete)):
    t = complete[j]
    rows = complete[:j]
    if len(rows) < 3:
        continue
    X = D[rows]
    vary = np.ones(X.shape[1], dtype=bool)
    vary[1:] = X[:, 1:].std(axis=0) > 0
    if vary.sum() > len(rows):
        continue
    Xk = X[:, vary]
    beta, _, rank, _ = np.linalg.lstsq(Xk, y[rows], rcond=None)
    if rank < Xk.shape[1]:  # empty until the past design has full rank
        continue
    resid[t] = y[t] - D[t, vary] @ beta

gas["resid"] = resid
gas["surprise7"] = gas["resid"].rolling(7, min_periods=7).sum()  # day t and the six preceding gas days
past = gas["surprise7"].shift(1)
gas["surprise7_z"] = (
    gas["surprise7"] - past.rolling(365, min_periods=365).mean()
) / past.rolling(365, min_periods=365).std()  # previous 365 gas days, excluding day t

In [20]:
gas[["surprise7", "surprise7_z"]] = gas[["surprise7", "surprise7_z"]].shift(2)  # gas day D is usable at D+2

In [21]:
px = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\2. Data Processing\prices_processed.csv", parse_dates=["date"])
funda_p = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\2. Data Processing\funda_processed.csv", parse_dates=["gasDayStart"])
lng = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\2. Data Processing\lng_processed.csv", parse_dates=["gasDayStart"])
wx_p = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\2. Data Processing\weather_hist_processed.csv", parse_dates=["date"])
fc = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\2. Data Processing\weather_forecast_processed.csv", parse_dates=["date"])

px = px[(px["date"] >= "2016-01-04") & (px["date"] <= "2026-10-07")].sort_values("date")
funda_p = funda_p.merge(gas[["gasDayStart", "surprise7", "surprise7_z"]], on="gasDayStart", how="left")
funda_p = funda_p.sort_values("gasDayStart")
lng = lng.sort_values("gasDayStart")
wx_p = wx_p.rename(columns={"date": "wx_date"}).sort_values("wx_date")

In [22]:
# Backward as-of, three calendar days. A Sunday print sticks to Monday. Nothing is carried further.
db = pd.merge_asof(px, funda_p, left_on="date", right_on="gasDayStart", direction="backward", tolerance=pd.Timedelta("3D"))
db = db.drop(columns=["gasDayStart"])
db = pd.merge_asof(db, lng, left_on="date", right_on="gasDayStart", direction="backward", tolerance=pd.Timedelta("3D"))
db = db.drop(columns=["gasDayStart"])
db = pd.merge_asof(db, wx_p, left_on="date", right_on="wx_date", direction="backward", tolerance=pd.Timedelta("3D"))
db = db.drop(columns=["wx_date"])

In [23]:
db = db.merge(fc, on="date", how="left")  # issue date; empty before 15 Feb 2024, do not fill
db = db.sort_values("date").reset_index(drop=True)

In [24]:
db = db[[
    "date", "roll_day", "dte",
    "m1", "m2", "m3", "s12", "s13", "s23", "s12_pct", "s13_pct", "s23_pct",
    "ds12_fwd5", "ds13_fwd5",
    "vol_m1", "vol_m2", "vol_m3", "sd_ds12", "sd_ds13",
    "ttf_switch", "switch_gap",
    "full", "full_norm5y", "storage_dev", "storage_dev_sq",
    "net_inj", "inj_util", "wd_util", "surprise7", "surprise7_z", "agsi_estimated",
    "hdd", "cdd", "hdd_anom7", "cdd_anom7", "fc_hdd_anom7",
    "lng7", "lng_dev", "sendout_bad",
    "d_feb", "d_mar", "d_apr", "d_may", "d_jun", "d_jul", "d_aug", "d_sep", "d_oct", "d_nov", "d_dec",
    "inj_season", "crisis",
]]

In [25]:
assert db["date"].is_monotonic_increasing and not db["date"].duplicated().any()
assert db["date"].min() == pd.Timestamp("2016-01-04") and db["date"].max() == pd.Timestamp("2026-10-07")
assert db["storage_dev"].notna().all()  # no unexplained null after the first session

pre = db["date"] < "2024-02-15"
span = (db["date"] >= "2024-02-15") & (db["date"] <= "2026-10-05")
assert db.loc[pre, "fc_hdd_anom7"].isna().all() and db.loc[span, "fc_hdd_anom7"].notna().all()

assert not {"consumption", "consumptionFull", "trend"} & set(db.columns)
targets = ["s12", "s13", "s23", "s12_pct", "s13_pct", "s23_pct", "ds12_fwd5", "ds13_fwd5"]
assert all(c in db.columns for c in targets) and all(c in db.columns for c in ["m1", "m2", "m3"])

In [26]:
db.to_csv("final_db.csv", index=False)